In [1]:
import os
import glob
import cv2
from ultralytics import YOLO

# モデル読み込み
model = YOLO("yolov8_weights.pt")
car_class_id = [k for k, v in model.names.items() if v == "car"][0]

# 入出力パス
input_dir = "./right_images/000"
output_dir = "./detections/000"
os.makedirs(output_dir, exist_ok=True)

# 対象画像取得
image_paths = sorted(glob.glob(os.path.join(input_dir, "*.png")))

# 検出処理
for image_path in image_paths:
    filename = os.path.basename(image_path)
    img = cv2.imread(image_path)

    results = model(img)[0]
    boxes = results.boxes

    if boxes is None or len(boxes) == 0:
        print(f"❌ No detection in {filename}")
        continue

    for box in boxes:
        cls_id = int(box.cls)
        conf = float(box.conf)
        if cls_id == car_class_id:
            x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
            label = f"car {conf:.2f}"
            cv2.putText(img, label, (x1, y1 - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

    # 検出付き画像を保存
    out_path = os.path.join(output_dir, filename)
    cv2.imwrite(out_path, img)
    print(f"✅ Saved detection: {filename}")



0: 288x640 1 person, 7 cars, 32.1ms
Speed: 10.5ms preprocess, 32.1ms inference, 39.7ms postprocess per image at shape (1, 3, 288, 640)
✅ Saved detection: frame_00001.png

0: 288x640 1 person, 7 cars, 13.1ms
Speed: 2.6ms preprocess, 13.1ms inference, 1.2ms postprocess per image at shape (1, 3, 288, 640)
✅ Saved detection: frame_00002.png

0: 288x640 1 person, 6 cars, 16.0ms
Speed: 1.7ms preprocess, 16.0ms inference, 1.3ms postprocess per image at shape (1, 3, 288, 640)
✅ Saved detection: frame_00003.png

0: 288x640 1 person, 7 cars, 13.8ms
Speed: 1.6ms preprocess, 13.8ms inference, 1.5ms postprocess per image at shape (1, 3, 288, 640)
✅ Saved detection: frame_00004.png

0: 288x640 2 persons, 6 cars, 11.3ms
Speed: 1.6ms preprocess, 11.3ms inference, 1.2ms postprocess per image at shape (1, 3, 288, 640)
✅ Saved detection: frame_00005.png

0: 288x640 2 persons, 7 cars, 1 truck, 12.2ms
Speed: 2.0ms preprocess, 12.2ms inference, 1.3ms postprocess per image at shape (1, 3, 288, 640)
✅ Saved 

In [1]:
import os
import glob
import cv2
from ultralytics import YOLO

# モデル読み込み
model = YOLO("yolov8_weights.pt")
car_class_id = [k for k, v in model.names.items() if v == "car"][0]

# 入出力パス
input_dir = "./right_images/000"
output_dir = "./detections/000"
os.makedirs(output_dir, exist_ok=True)

# 対象画像
image_paths = sorted(glob.glob(os.path.join(input_dir, "*.png")))

# 処理
for image_path in image_paths:
    filename = os.path.basename(image_path)
    img = cv2.imread(image_path)
    h, w = img.shape[:2]
    center_x = w // 2

    results = model(img)[0]
    boxes = results.boxes

    if boxes is None or len(boxes) == 0:
        print(f"❌ No detection in {filename}")
        continue

    # 中央に最も近いcarを選ぶ
    car_candidates = []
    for box in boxes:
        cls_id = int(box.cls)
        conf = float(box.conf)
        if cls_id != car_class_id:
            continue
        x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())
        bbox_center = (x1 + x2) / 2
        dist_to_center = abs(bbox_center - center_x)
        car_candidates.append((dist_to_center, conf, (x1, y1, x2, y2)))

    if not car_candidates:
        print(f"🚫 No car found in {filename}")
        continue

    # 最も中央寄りのcarを選ぶ
    _, best_conf, (x1, y1, x2, y2) = sorted(car_candidates, key=lambda x: (x[0], -x[1]))[0]

    # 可視化
    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
    label = f"car {best_conf:.2f}"
    cv2.putText(img, label, (x1, y1 - 5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

    # 保存
    out_path = os.path.join(output_dir, filename)
    cv2.imwrite(out_path, img)
    print(f"✅ 1 car saved: {filename}")



0: 288x640 1 person, 7 cars, 23.0ms
Speed: 3.7ms preprocess, 23.0ms inference, 3.3ms postprocess per image at shape (1, 3, 288, 640)
✅ 1 car saved: frame_00001.png

0: 288x640 1 person, 7 cars, 12.0ms
Speed: 1.9ms preprocess, 12.0ms inference, 1.2ms postprocess per image at shape (1, 3, 288, 640)
✅ 1 car saved: frame_00002.png

0: 288x640 1 person, 6 cars, 11.5ms
Speed: 1.7ms preprocess, 11.5ms inference, 1.3ms postprocess per image at shape (1, 3, 288, 640)
✅ 1 car saved: frame_00003.png

0: 288x640 1 person, 7 cars, 13.6ms
Speed: 2.3ms preprocess, 13.6ms inference, 1.2ms postprocess per image at shape (1, 3, 288, 640)
✅ 1 car saved: frame_00004.png

0: 288x640 2 persons, 6 cars, 12.3ms
Speed: 1.9ms preprocess, 12.3ms inference, 1.2ms postprocess per image at shape (1, 3, 288, 640)
✅ 1 car saved: frame_00005.png

0: 288x640 2 persons, 7 cars, 1 truck, 12.9ms
Speed: 1.8ms preprocess, 12.9ms inference, 1.3ms postprocess per image at shape (1, 3, 288, 640)
✅ 1 car saved: frame_00006.png

In [2]:
import os
import glob
import cv2
from ultralytics import YOLO

# モデル読み込み
model = YOLO("yolov8_weights.pt")
car_class_id = [k for k, v in model.names.items() if v == "car"][0]

# 入出力ディレクトリ
input_root = "./right_images"
output_root = "./detections"
os.makedirs(output_root, exist_ok=True)

# 処理対象scene
scene_ids = [f"{i:03d}" for i in range(2, 11)]  # "002" ～ "010"

for scene_id in scene_ids:
    input_dir = os.path.join(input_root, scene_id)
    output_dir = os.path.join(output_root, scene_id)
    os.makedirs(output_dir, exist_ok=True)

    image_paths = sorted(glob.glob(os.path.join(input_dir, "*.png")))

    for image_path in image_paths:
        filename = os.path.basename(image_path)
        img = cv2.imread(image_path)
        h, w = img.shape[:2]
        center_x = w // 2

        results = model(img)[0]
        boxes = results.boxes

        if boxes is None or len(boxes) == 0:
            print(f"❌ No detection in {scene_id}/{filename}")
            continue

        car_candidates = []
        for box in boxes:
            cls_id = int(box.cls)
            conf = float(box.conf)
            if cls_id != car_class_id:
                continue
            x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())
            bbox_center = (x1 + x2) / 2
            dist_to_center = abs(bbox_center - center_x)
            car_candidates.append((dist_to_center, conf, (x1, y1, x2, y2)))

        if not car_candidates:
            print(f"🚫 No car found in {scene_id}/{filename}")
            continue

        # 中央に最も近い1台のみ
        _, best_conf, (x1, y1, x2, y2) = sorted(car_candidates, key=lambda x: (x[0], -x[1]))[0]

        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        label = f"car {best_conf:.2f}"
        cv2.putText(img, label, (x1, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

        out_path = os.path.join(output_dir, filename)
        cv2.imwrite(out_path, img)
        print(f"✅ Saved: {scene_id}/{filename}")



0: 288x640 2 cars, 14.2ms
Speed: 1.6ms preprocess, 14.2ms inference, 1.4ms postprocess per image at shape (1, 3, 288, 640)
✅ Saved: 002/frame_00001.png

0: 288x640 2 cars, 12.0ms
Speed: 1.7ms preprocess, 12.0ms inference, 1.2ms postprocess per image at shape (1, 3, 288, 640)
✅ Saved: 002/frame_00002.png

0: 288x640 1 person, 3 cars, 13.6ms
Speed: 1.8ms preprocess, 13.6ms inference, 1.5ms postprocess per image at shape (1, 3, 288, 640)
✅ Saved: 002/frame_00003.png

0: 288x640 4 cars, 12.6ms
Speed: 1.6ms preprocess, 12.6ms inference, 1.7ms postprocess per image at shape (1, 3, 288, 640)
✅ Saved: 002/frame_00004.png

0: 288x640 2 cars, 11.8ms
Speed: 1.5ms preprocess, 11.8ms inference, 1.2ms postprocess per image at shape (1, 3, 288, 640)
✅ Saved: 002/frame_00005.png

0: 288x640 3 cars, 11.2ms
Speed: 1.6ms preprocess, 11.2ms inference, 1.2ms postprocess per image at shape (1, 3, 288, 640)
✅ Saved: 002/frame_00006.png

0: 288x640 1 car, 15.2ms
Speed: 2.2ms preprocess, 15.2ms inference, 1.3

KeyboardInterrupt: 

In [3]:
import os
import glob
import cv2
from ultralytics import YOLO

# モデル読み込み
model = YOLO("yolov8_weights.pt")
car_class_id = [k for k, v in model.names.items() if v == "car"][0]

# 入出力ルート
input_root = "./right_images"
output_root = "./detections"
os.makedirs(output_root, exist_ok=True)

# ログファイル
error_log = open("error_images.txt", "w")
no_car_log = open("no_car_detected.txt", "w")

# 処理対象scene
scene_ids = [f"{i:03d}" for i in range(2, 11)]

for scene_id in scene_ids:
    input_dir = os.path.join(input_root, scene_id)
    output_dir = os.path.join(output_root, scene_id)
    os.makedirs(output_dir, exist_ok=True)

    image_paths = sorted(glob.glob(os.path.join(input_dir, "*.png")))

    for image_path in image_paths:
        filename = os.path.basename(image_path)
        try:
            img = cv2.imread(image_path)
            if img is None:
                raise ValueError("Image load failed")

            h, w = img.shape[:2]
            center_x = w // 2

            results = model(img)[0]
            boxes = results.boxes

            if boxes is None or len(boxes) == 0:
                print(f"❌ No detection in {scene_id}/{filename}")
                no_car_log.write(f"{scene_id}/{filename}\n")
                continue

            car_candidates = []
            for box in boxes:
                cls_id = int(box.cls)
                conf = float(box.conf)
                if cls_id != car_class_id:
                    continue
                x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())
                bbox_center = (x1 + x2) / 2
                dist_to_center = abs(bbox_center - center_x)
                car_candidates.append((dist_to_center, conf, (x1, y1, x2, y2)))

            if not car_candidates:
                print(f"🚫 No car found in {scene_id}/{filename}")
                no_car_log.write(f"{scene_id}/{filename}\n")
                continue

            _, best_conf, (x1, y1, x2, y2) = sorted(car_candidates, key=lambda x: (x[0], -x[1]))[0]

            # 描画
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
            label = f"car {best_conf:.2f}"
            cv2.putText(img, label, (x1, y1 - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

            out_path = os.path.join(output_dir, filename)
            cv2.imwrite(out_path, img)
            print(f"✅ Saved: {scene_id}/{filename}")

        except Exception as e:
            print(f"⚠️ Error in {scene_id}/{filename}: {e}")
            error_log.write(f"{scene_id}/{filename} - {e}\n")

# ログファイルを閉じる
error_log.close()
no_car_log.close()



0: 288x640 2 cars, 12.7ms
Speed: 1.6ms preprocess, 12.7ms inference, 1.2ms postprocess per image at shape (1, 3, 288, 640)
✅ Saved: 002/frame_00001.png

0: 288x640 2 cars, 23.4ms
Speed: 2.4ms preprocess, 23.4ms inference, 1.6ms postprocess per image at shape (1, 3, 288, 640)
✅ Saved: 002/frame_00002.png

0: 288x640 1 person, 3 cars, 13.6ms
Speed: 1.5ms preprocess, 13.6ms inference, 1.5ms postprocess per image at shape (1, 3, 288, 640)
✅ Saved: 002/frame_00003.png

0: 288x640 4 cars, 12.9ms
Speed: 1.6ms preprocess, 12.9ms inference, 1.5ms postprocess per image at shape (1, 3, 288, 640)
✅ Saved: 002/frame_00004.png

0: 288x640 2 cars, 13.3ms
Speed: 1.5ms preprocess, 13.3ms inference, 1.5ms postprocess per image at shape (1, 3, 288, 640)
✅ Saved: 002/frame_00005.png

0: 288x640 3 cars, 13.8ms
Speed: 1.5ms preprocess, 13.8ms inference, 1.4ms postprocess per image at shape (1, 3, 288, 640)
✅ Saved: 002/frame_00006.png

0: 288x640 1 car, 13.2ms
Speed: 1.6ms preprocess, 13.2ms inference, 1.3

In [4]:
import os
import cv2
from ultralytics import YOLO

# 再処理対象の画像一覧（scene_id/filename）
target_images = [
    "003/frame_00001.png", "003/frame_00002.png", "003/frame_00003.png",
    "003/frame_00004.png", "003/frame_00005.png", "003/frame_00006.png",
    "003/frame_00007.png", "003/frame_00008.png", "003/frame_00009.png",
    "003/frame_00010.png", "003/frame_00011.png", "003/frame_00012.png",
    "003/frame_00013.png", "003/frame_00014.png", "003/frame_00015.png",
    "003/frame_00017.png", "003/frame_00019.png", "003/frame_00020.png",
    "003/frame_00021.png", "003/frame_00022.png", "003/frame_00035.png",
    "003/frame_00053.png", "003/frame_00055.png", "003/frame_00056.png",
    "003/frame_00057.png", "003/frame_00066.png", "003/frame_00067.png",
    "003/frame_00068.png", "003/frame_00071.png", "003/frame_00072.png",
    "003/frame_00074.png", "003/frame_00075.png", "003/frame_00076.png",
    "003/frame_00077.png", "003/frame_00078.png", "003/frame_00079.png",
    "003/frame_00081.png", "003/frame_00083.png", "003/frame_00084.png",
    "003/frame_00085.png", "003/frame_00086.png", "003/frame_00087.png",
    "003/frame_00088.png", "003/frame_00089.png", "003/frame_00090.png",
    "003/frame_00091.png", "003/frame_00092.png", "003/frame_00095.png",
    "003/frame_00097.png", "003/frame_00098.png", "003/frame_00099.png",
    "003/frame_00100.png", "003/frame_00101.png", "003/frame_00102.png",
    "003/frame_00103.png", "003/frame_00104.png", "003/frame_00105.png",
    "003/frame_00106.png", "003/frame_00116.png", "003/frame_00117.png",
    "003/frame_00118.png", "003/frame_00119.png", "003/frame_00120.png",
    "003/frame_00122.png", "003/frame_00123.png", "003/frame_00124.png",
    "003/frame_00125.png", "003/frame_00126.png", "003/frame_00127.png",
    "003/frame_00128.png", "003/frame_00129.png", "003/frame_00130.png",
    "003/frame_00131.png", "003/frame_00132.png", "003/frame_00133.png",
    "003/frame_00134.png", "003/frame_00139.png", "003/frame_00140.png",
    "003/frame_00141.png", "003/frame_00142.png", "003/frame_00143.png",
    "003/frame_00145.png", "003/frame_00146.png", "003/frame_00147.png",
    "003/frame_00148.png", "003/frame_00149.png", "003/frame_00150.png",
    "003/frame_00152.png", "003/frame_00154.png", "003/frame_00156.png",
    "003/frame_00157.png", "003/frame_00158.png", "003/frame_00159.png",
    "004/frame_00001.png", "004/frame_00003.png", "004/frame_00004.png",
    "004/frame_00005.png", "004/frame_00006.png", "004/frame_00009.png",
    "004/frame_00017.png", "004/frame_00025.png", "004/frame_00030.png",
    "004/frame_00042.png", "004/frame_00058.png", "004/frame_00063.png",
    "004/frame_00080.png", "004/frame_00082.png", "004/frame_00083.png",
    "010/frame_00083.png", "010/frame_00085.png", "010/frame_00089.png",
    "010/frame_00090.png", "010/frame_00091.png", "010/frame_00092.png",
    "010/frame_00093.png", "010/frame_00094.png", "010/frame_00095.png",
    "010/frame_00096.png", "010/frame_00097.png", "010/frame_00098.png",
    "010/frame_00099.png", "010/frame_00100.png", "010/frame_00101.png",
    "010/frame_00103.png", "010/frame_00104.png", "010/frame_00105.png",
    "010/frame_00107.png"
]


# モデル読み込み
model = YOLO("yolov8_weights.pt")
car_class_id = [k for k, v in model.names.items() if v == "car"][0]

# ディレクトリルート
input_root = "./right_images"
output_root = "./detections"
os.makedirs(output_root, exist_ok=True)

for path in target_images:
    scene_id, filename = path.split("/")
    input_path = os.path.join(input_root, scene_id, filename)
    output_dir = os.path.join(output_root, scene_id)
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, filename)

    try:
        img = cv2.imread(input_path)
        if img is None:
            print(f"❌ 読み込み失敗: {path}")
            continue

        h, w = img.shape[:2]
        center_x = w // 2

        results = model(img)[0]
        boxes = results.boxes

        if boxes is None or len(boxes) == 0:
            print(f"🚫 検出失敗: {path}")
            continue

        # carのみを抽出
        car_candidates = []
        for box in boxes:
            cls_id = int(box.cls)
            conf = float(box.conf)
            if cls_id != car_class_id:
                continue
            x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())
            bbox_center = (x1 + x2) / 2
            dist_to_center = abs(bbox_center - center_x)
            car_candidates.append((dist_to_center, conf, (x1, y1, x2, y2)))

        if not car_candidates:
            print(f"🚫 carなし: {path}")
            continue

        _, best_conf, (x1, y1, x2, y2) = sorted(car_candidates, key=lambda x: (x[0], -x[1]))[0]

        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        label = f"car {best_conf:.2f}"
        cv2.putText(img, label, (x1, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

        cv2.imwrite(output_path, img)
        print(f"✅ 再検出成功: {path}")

    except Exception as e:
        print(f"⚠️ 処理中エラー: {path} → {e}")



0: 288x640 1 person, 1 motorcycle, 4 buss, 12.3ms
Speed: 1.6ms preprocess, 12.3ms inference, 1.2ms postprocess per image at shape (1, 3, 288, 640)
🚫 carなし: 003/frame_00001.png

0: 288x640 1 person, 1 motorcycle, 3 buss, 12.6ms
Speed: 1.5ms preprocess, 12.6ms inference, 1.3ms postprocess per image at shape (1, 3, 288, 640)
🚫 carなし: 003/frame_00002.png

0: 288x640 1 person, 1 motorcycle, 2 buss, 2 trucks, 12.2ms
Speed: 1.7ms preprocess, 12.2ms inference, 1.2ms postprocess per image at shape (1, 3, 288, 640)
🚫 carなし: 003/frame_00003.png

0: 288x640 1 motorcycle, 4 buss, 1 truck, 13.6ms
Speed: 1.6ms preprocess, 13.6ms inference, 1.1ms postprocess per image at shape (1, 3, 288, 640)
🚫 carなし: 003/frame_00004.png

0: 288x640 3 buss, 14.0ms
Speed: 1.6ms preprocess, 14.0ms inference, 1.2ms postprocess per image at shape (1, 3, 288, 640)
🚫 carなし: 003/frame_00005.png

0: 288x640 1 person, 3 buss, 12.0ms
Speed: 1.7ms preprocess, 12.0ms inference, 1.1ms postprocess per image at shape (1, 3, 288, 6